In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

In [2]:
input_gpkg = Path("../src/osm_x_usgs_x_naacc_36001_database.gpkg")
output_gpx = Path("./osm_ways_x_nhd_flowlines_x_naacc_crossings.gpx")

In [3]:
gdf = gpd.read_file(
    filename=input_gpkg, #
    layer="osm_ways_x_nhd_flowlines_x_naacc_crossings",
    engine="pyogrio"
)

In [17]:
import pandas as pd
import geopandas as gpd
from collections import OrderedDict

# Assume 'gdf' is your original GeoDataFrame


def create_description(row):
    """Creates an indented text block with a Google Maps link at the top."""

    columns_to_include = [
        "osm_road_name",
        "osm_from_name",
        "osm_to_name",
        "naacc_road_name",
        "naacc_crossing_comment",
        "naacc_location_description",
        "naacc_stream_name",
        "nhd_waterway_name",
        "match_distance_m",
        "match_confidence_score",
        "naacc_crossing_code",
        "naacc_crossing_type",
        "naacc_inlet_structure_type",
        "naacc_outlet_structure_type",
        "naacc_structure_comment",
        "nhd_flowline_permanent_identifier",
        "nhd_waterway_type",
        "nhd_waterway_type_description",
        "nhd_stream_type_description",
        "nhd_mean_annual_gage_adjusted_flow_cu_ft_per_sec",
        "match_score",
        "match_type",
        "osm_road_type",
        "osm_is_roadway",
        "osm_is_service_road",
        "naacc_is_trail",
        "naacc_is_unnamed_road",
        "naacc_is_driveway",
    ]


    # Get latitude and longitude from the geometry
    lat = row.geometry.y
    lon = row.geometry.x

    # Create the Google Maps link
    gmaps_link = f"https://www.google.com/maps?q={lat},{lon}"
    gmaps_link = f"https://www.google.com/maps?q={lat},{lon}"

    # Start the description with the link and a separator
    lines = [
        f"Location: ({lat}, {lon})",
        f"Google Maps Link\n    {gmaps_link} ",
        "---"
    ]

    # Define which columns to include in the rest of the description

    for col in columns_to_include:
        if col in row.index:
            value = row[col]
            display_value = value if pd.notna(value) else "N/A"
            # Format as:
            # Key
            #     Value
            lines.append(f"\n• {col}\n        {display_value}")

    # Join all blocks together with a blank line in between
    return "\n".join(lines)

In [18]:

# 1. Apply the new function
descriptions = gdf.apply(create_description, axis=1)

# 2. Prepare the clean data dictionary for export
export_data = {
    "geometry": gdf.geometry,
    "name": gdf["naacc_crossing_code"].astype(str),
    "desc": descriptions,
    "ele": 0,
}

# 3. Create the final GeoDataFrame for export
export_gdf = gpd.GeoDataFrame(export_data, crs=gdf.crs)

# 4. Define the schema for the writer
schema = {
    "geometry": "Point",
    "properties": OrderedDict([("name", "str"), ("desc", "str"), ("ele", "float")]),
}

# 5. Export using fiona and the defined schema
export_gdf.to_file(filename=output_gpx, driver="GPX", engine="fiona", schema=schema)

print(f"Successfully exported to {output_gpx}")


Successfully exported to osm_ways_x_nhd_flowlines_x_naacc_crossings.gpx


In [10]:
export_gdf.iloc[0]["desc"]

'Google Maps Link\n    www.google.com/maps?q=42.43028925376134,-74.23622446791472 \n---\n◦ osm_road_name\n    CR 362/County Route 362\n◦ osm_from_name\n    Knowles Road\n◦ osm_to_name\n    Smith Road\n◦ naacc_road_name\n    362\n◦ naacc_crossing_comment\n    N/A\n◦ naacc_location_description\n    Near guide rails\n◦ naacc_stream_name\n    N/A\n◦ nhd_waterway_name\n    N/A\n◦ match_distance_m\n    0.7737371416413319\n◦ match_confidence_score\n    83.0\n◦ naacc_crossing_code\n    xy4243052774236313\n◦ naacc_crossing_type\n    Culvert\n◦ naacc_inlet_structure_type\n    Round Culvert\n◦ naacc_outlet_structure_type\n    Round Culvert\n◦ naacc_structure_comment\n    No data\n◦ nhd_flowline_permanent_identifier\n    N/A\n◦ nhd_waterway_type\n    N/A\n◦ nhd_waterway_type_description\n    N/A\n◦ nhd_stream_type_description\n    N/A\n◦ nhd_mean_annual_gage_adjusted_flow_cu_ft_per_sec\n    N/A\n◦ match_score\n    0.7520725016753746\n◦ match_type\n    shortest_line_to_road_span\n◦ osm_road_type\n 